# Direct Trade Booking Via RabbitMQ

This notebook bypasses `helix-rest` and behaves like an external blotter or booking system:

1. insert a trade directly into the Helix SQLite store
2. publish `trade.compute`
3. publish `position.pl.compute`

The running `helix-runtime` service then computes notional, positions, portfolio P&L, and portfolio risk, publishes Kafka updates, and the Helix UI refreshes through SSE.

Prerequisites:

- `./scripts/linux/launch.sh` or the Windows equivalent is already running
- RabbitMQ is reachable on `localhost:5672`
- the selected `portfolio_id`, `instrument_id`, and `book` already exist in the seeded reference data
- `pika` is installed in the Python environment used by Jupyter


In [1]:
from __future__ import annotations

import json
import os
import sqlite3
from dataclasses import dataclass
from datetime import UTC, datetime, timedelta
from pathlib import Path
from uuid import uuid4

import pika

In [2]:
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DB_PATH = Path(os.environ.get("HELIX_DB_PATH", REPO_ROOT / "helix-store" / "helix.db"))
RABBITMQ_HOST = os.environ.get("HELIX_RABBITMQ_HOST", "localhost")
RABBITMQ_PORT = int(os.environ.get("HELIX_RABBITMQ_PORT", "5672"))
RABBITMQ_USERNAME = os.environ.get("HELIX_RABBITMQ_USERNAME", "guest")
RABBITMQ_PASSWORD = os.environ.get("HELIX_RABBITMQ_PASSWORD", "guest")
RABBITMQ_VHOST = os.environ.get("HELIX_RABBITMQ_VHOST", "/")

TRADE_COMPUTE_QUEUE = os.environ.get("HELIX_RABBITMQ_QUEUE_TRADE_COMPUTE", "trade.compute")
POSITION_PL_COMPUTE_QUEUE = os.environ.get("HELIX_RABBITMQ_QUEUE_POSITION_PL_COMPUTE", "position.pl.compute")

print(f"DB_PATH={DB_PATH}")
print(f"RabbitMQ={RABBITMQ_HOST}:{RABBITMQ_PORT} vhost={RABBITMQ_VHOST}")
print(f"Queues={TRADE_COMPUTE_QUEUE}, {POSITION_PL_COMPUTE_QUEUE}")

DB_PATH=/Users/alexandershubert/git/helix/helix-store/helix.db
RabbitMQ=localhost:5672 vhost=/
Queues=trade.compute, position.pl.compute


In [3]:
@dataclass(frozen=True)
class TradeBookingRequest:
    portfolio_id: str
    instrument_id: str
    side: str
    quantity: float
    price: float
    settlement_date: str
    book: str


def utc_now() -> datetime:
    return datetime.now(UTC)


def isoformat_utc(value: datetime) -> str:
    return value.astimezone(UTC).isoformat().replace("+00:00", "Z")


def build_trade_identifiers(portfolio_id: str, submitted_at: datetime) -> tuple[str, str]:
    stamp = submitted_at.strftime("%Y%m%d%H%M%S%f")[:-3]
    return (f"TRD-{portfolio_id}-{stamp}", f"{portfolio_id}-POS-{stamp}")


def validate_reference_data(conn: sqlite3.Connection, request: TradeBookingRequest) -> dict[str, str]:
    portfolio_row = conn.execute(
        "SELECT portfolio_id FROM portfolio WHERE portfolio_id = ?",
        (request.portfolio_id,),
    ).fetchone()
    if portfolio_row is None:
        raise ValueError(f"Unknown portfolio_id: {request.portfolio_id}")

    instrument_row = conn.execute(
        """
        SELECT instrument_id, instrument_name, asset_class, currency
        FROM instrument
        WHERE instrument_id = ? AND active = 1
        """,
        (request.instrument_id,),
    ).fetchone()
    if instrument_row is None:
        raise ValueError(f"Unknown active instrument_id: {request.instrument_id}")

    book_row = conn.execute(
        "SELECT name FROM book WHERE name = ?",
        (request.book,),
    ).fetchone()
    if book_row is None:
        raise ValueError(f"Unknown book: {request.book}")

    return {
        "instrument_name": str(instrument_row[1]),
        "asset_class": str(instrument_row[2]),
        "currency": str(instrument_row[3]),
    }


def insert_trade(conn: sqlite3.Connection, request: TradeBookingRequest, submitted_at: datetime) -> str:
    side = request.side.upper()
    if side not in {"BUY", "SELL"}:
        raise ValueError("side must be BUY or SELL")

    reference = validate_reference_data(conn, request)
    trade_id, position_id = build_trade_identifiers(request.portfolio_id, submitted_at)
    trade_timestamp = isoformat_utc(submitted_at)

    conn.execute(
        """
        INSERT INTO trades (
          trade_id, portfolio_id, position_id, instrument_id, instrument_name,
          asset_class, currency, side, quantity, price, notional, trade_timestamp,
          settlement_date, book, status, version, created_at, updated_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            trade_id,
            request.portfolio_id,
            position_id,
            request.instrument_id,
            reference["instrument_name"],
            reference["asset_class"],
            reference["currency"],
            side,
            request.quantity,
            request.price,
            None,
            trade_timestamp,
            request.settlement_date,
            request.book,
            "accepted",
            1,
            trade_timestamp,
            trade_timestamp,
        ),
    )
    conn.commit()
    return trade_id


def build_task_payload(task_type: str, portfolio_id: str, requested_at: datetime, source_event_id: str | None = None) -> dict[str, object]:
    payload = {
        "taskId": f"TASK-{uuid4().hex[:12].upper()}",
        "taskType": task_type,
        "portfolioId": portfolio_id,
        "requestedAt": isoformat_utc(requested_at),
    }
    if source_event_id:
        payload["sourceEventId"] = source_event_id
    return payload


def publish_tasks(portfolio_id: str, trade_id: str, requested_at: datetime) -> None:
    credentials = pika.PlainCredentials(RABBITMQ_USERNAME, RABBITMQ_PASSWORD)
    parameters = pika.ConnectionParameters(
        host=RABBITMQ_HOST,
        port=RABBITMQ_PORT,
        virtual_host=RABBITMQ_VHOST,
        credentials=credentials,
    )
    with pika.BlockingConnection(parameters) as connection:
        channel = connection.channel()
        for queue_name, source_event_id in [
            (TRADE_COMPUTE_QUEUE, trade_id),
            (POSITION_PL_COMPUTE_QUEUE, trade_id),
        ]:
            channel.queue_declare(queue=queue_name, durable=True)
            payload = build_task_payload(queue_name, portfolio_id, requested_at, source_event_id)
            channel.basic_publish(
                exchange="",
                routing_key=queue_name,
                body=json.dumps(payload).encode("utf-8"),
                properties=pika.BasicProperties(content_type="application/json", delivery_mode=2),
            )
            print(f"published {queue_name}: {payload}")


def book_trade_direct(request: TradeBookingRequest) -> str:
    submitted_at = utc_now()
    with sqlite3.connect(DB_PATH) as conn:
        trade_id = insert_trade(conn, request, submitted_at)
    publish_tasks(request.portfolio_id, trade_id, submitted_at)
    print(f"inserted trade_id={trade_id}")
    return trade_id

In [7]:
# Example request. Adjust as needed.
request = TradeBookingRequest(
    portfolio_id="PF-EQ",
    instrument_id="AAPL",
    side="SELL",
    quantity=10.0,
    price=800.5,
    settlement_date=(utc_now() + timedelta(days=1)).date().isoformat(),
    book="EQ-789",
)

request

TradeBookingRequest(portfolio_id='PF-EQ', instrument_id='AAPL', side='SELL', quantity=10.0, price=800.5, settlement_date='2026-03-23', book='EQ-789')

In [8]:
trade_id = book_trade_direct(request)
trade_id

published trade.compute: {'taskId': 'TASK-F2BFF6425DB5', 'taskType': 'trade.compute', 'portfolioId': 'PF-EQ', 'requestedAt': '2026-03-22T10:43:56.364225Z', 'sourceEventId': 'TRD-PF-EQ-20260322104356364'}
published position.pl.compute: {'taskId': 'TASK-D44E8FC16794', 'taskType': 'position.pl.compute', 'portfolioId': 'PF-EQ', 'requestedAt': '2026-03-22T10:43:56.364225Z', 'sourceEventId': 'TRD-PF-EQ-20260322104356364'}
inserted trade_id=TRD-PF-EQ-20260322104356364


'TRD-PF-EQ-20260322104356364'

In [6]:
# Optional: confirm the trade row is present and see the latest snapshots written by runtime.
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row

    trade_row = conn.execute(
        "SELECT trade_id, portfolio_id, instrument_id, side, quantity, price, notional, status, updated_at FROM trades WHERE trade_id = ?",
        (trade_id,),
    ).fetchone()

    position_rows = conn.execute(
        """
        SELECT instrument_id, quantity, direction, realized_pnl, unrealized_pnl, total_pnl, as_of_ts
        FROM position
        WHERE portfolio_id = ?
        ORDER BY as_of_ts DESC, instrument_id ASC
        LIMIT 10
        """,
        (request.portfolio_id,),
    ).fetchall()

    pnl_row = conn.execute(
        "SELECT total_pnl, realized_pnl, unrealized_pnl, valuation_ts FROM pnl WHERE portfolio_id = ? ORDER BY valuation_ts DESC LIMIT 1",
        (request.portfolio_id,),
    ).fetchone()

    risk_row = conn.execute(
        "SELECT delta, gross_exposure, net_exposure, var_95, valuation_ts FROM risk WHERE portfolio_id = ? ORDER BY valuation_ts DESC LIMIT 1",
        (request.portfolio_id,),
    ).fetchone()

display(dict(trade_row) if trade_row else None)
display([dict(row) for row in position_rows])
display(dict(pnl_row) if pnl_row else None)
display(dict(risk_row) if risk_row else None)

{'trade_id': 'TRD-PF-EQ-20260322104322432',
 'portfolio_id': 'PF-EQ',
 'instrument_id': 'AAPL',
 'side': 'BUY',
 'quantity': 25.0,
 'price': 201.5,
 'notional': 5037.5,
 'status': 'processed',
 'updated_at': '2026-03-22T10:43:22.432943Z'}

[{'instrument_id': 'AAPL',
  'quantity': 25.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': -237.5,
  'total_pnl': -237.5,
  'as_of_ts': '2026-03-22T10:43:22.432943Z'},
 {'instrument_id': 'NVDA',
  'quantity': 15.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': 75.0,
  'total_pnl': 75.0,
  'as_of_ts': '2026-03-22T10:43:22.432943Z'},
 {'instrument_id': 'NVDA',
  'quantity': 15.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': 75.0,
  'total_pnl': 75.0,
  'as_of_ts': '2026-03-22T10:43:08.447160Z'},
 {'instrument_id': 'AAPL',
  'quantity': 10.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': -80.0,
  'total_pnl': -80.0,
  'as_of_ts': '2026-03-22T10:37:06.042587Z'},
 {'instrument_id': 'NVDA',
  'quantity': 15.0,
  'direction': 'LONG',
  'realized_pnl': 0.0,
  'unrealized_pnl': 75.0,
  'total_pnl': 75.0,
  'as_of_ts': '2026-03-22T10:37:06.042587Z'},
 {'instrument_id': 'AAPL',
  'quantity': 10.0,
  'direction': 'L

{'total_pnl': -162.5,
 'realized_pnl': 0.0,
 'unrealized_pnl': -162.5,
 'valuation_ts': '2026-03-22T10:43:22.432943Z'}

{'delta': 15450.0,
 'gross_exposure': 15450.0,
 'net_exposure': 15450.0,
 'var_95': 6461.23,
 'valuation_ts': '2026-03-22T10:43:22.432943Z'}